In [ ]:
import numpy as np
import pickle

from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Bidirectional, RepeatVector, Dropout, TimeDistributed
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.losses import sparse_categorical_crossentropy

## Load data

## Encoder decoder
Most of the code is copied from the following tutorial:
https://www.tensorflow.org/addons/tutorials/networks_seq2seq_nmt

In [2]:
import tensorflow as tf
import tensorflow_addons as tfa
import time

### Load and preprocess the data

In [3]:
with open('data/preprocessed/dicts.pkl', 'rb') as fin:
    conversion_dicts = pickle.load(fin)

In [4]:
with open('data/preprocessed/input_seqs_train.txt') as fin:
    input_seqs_train = fin.readlines()
    input_seqs_train = [s.strip() for s in input_seqs_train]
        
with open('data/preprocessed/output_seqs_train.txt') as fin:
    output_seqs_train = fin.readlines()
    output_seqs_train = [s.strip() for s in output_seqs_train]
        
with open('data/preprocessed/input_seqs_test.txt') as fin:
    input_seqs_test = fin.readlines()
    input_seqs_test = [s.strip() for s in input_seqs_test]
        
with open('data/preprocessed/output_seqs_test.txt') as fin:
    output_seqs_test = fin.readlines()
    output_seqs_test = [s.strip() for s in output_seqs_test]

for now, only take unique input-output combinations. This reduces the dataset quite a bit.

In [5]:
input_seqs_train_uq, output_seqs_train_uq = zip(*set(list(zip(input_seqs_train, output_seqs_train))))

In [6]:
input_seqs_test_uq, output_seqs_test_uq = zip(*set(list(zip(input_seqs_test, output_seqs_test))))

In [7]:
len(input_seqs_train), len(input_seqs_train_uq)

(269539, 49402)

In [8]:
## Add <start> and <end> chars
if '<start>' not in conversion_dicts['input_char2idx']:
    idx = len(conversion_dicts['input_char2idx'])
    conversion_dicts['input_char2idx']['<start>'] = idx
    conversion_dicts['input_idx2char'][idx] = '<start>'

if '<end>' not in conversion_dicts['input_char2idx']:
    idx = len(conversion_dicts['input_char2idx'])
    conversion_dicts['input_char2idx']['<end>'] = idx
    conversion_dicts['input_idx2char'][idx] = '<end>'
    
if '<start>' not in conversion_dicts['output_char2idx']:
    idx = len(conversion_dicts['output_char2idx'])
    conversion_dicts['output_char2idx']['<start>'] = idx
    conversion_dicts['output_idx2char'][idx] = '<start>'

if '<end>' not in conversion_dicts['output_char2idx']:
    idx = len(conversion_dicts['output_char2idx'])
    conversion_dicts['output_char2idx']['<end>'] = idx
    conversion_dicts['output_idx2char'][idx] = '<end>'

In [9]:
def preprocess_sequences(seqs, char2idx, max_len):
    seqs = [['<start>'] + [c for c in seq] + ['<end>'] for seq in seqs]
    inputs = [[char2idx[c] for c in seq] for seq in seqs]
    inputs = tf.keras.preprocessing.sequence.pad_sequences(inputs,
                                                          maxlen=max_len,
                                                          padding='post',
                                                          value=char2idx['PAD'])
    return inputs

In [10]:
max_length_input = max([len(s) for s in input_seqs_train]) + 2
max_length_output = max([len(s) for s in output_seqs_train]) + 2

In [11]:
X_train = preprocess_sequences(input_seqs_train_uq, conversion_dicts['input_char2idx'], max_length_input)
y_train = preprocess_sequences(output_seqs_train_uq, conversion_dicts['output_char2idx'], max_length_output)

In [12]:
X_train.shape, y_train.shape

((49402, 25), (49402, 26))

In [13]:
# Parameters
vocab_input_size = len(conversion_dicts['input_char2idx'].keys())
vocab_output_size = len(conversion_dicts['output_char2idx'].keys())
max_length_input = X_train.shape[1]
max_length_output = y_train.shape[1]

BATCH_SIZE = 32
embedding_dim = 30
hidden_dim = 500


In [27]:
import tensorflow as tf

class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)

        ##________ LSTM layer in Encoder ------- ##
        self.lstm_layer = tf.keras.layers.LSTM(self.enc_units,
                                       return_sequences=True,
                                       return_state=True,
                                       recurrent_initializer='glorot_uniform')



    def call(self, x, hidden):
        x = self.embedding(x)
        output, h, c = self.lstm_layer(x, initial_state = hidden)
        return output, h, c

    def initialize_hidden_state(self):
        return [tf.zeros((self.batch_sz, self.enc_units)), tf.zeros((self.batch_sz, self.enc_units))]

In [28]:
encoder = Encoder(vocab_input_size, 
                  embedding_dim=embedding_dim, 
                  enc_units=hidden_dim,
                 batch_sz = BATCH_SIZE)

# sample input
sample_hidden = encoder.initialize_hidden_state()
sample_output, sample_h, sample_c = encoder(X_train[:BATCH_SIZE], sample_hidden)
print ('Encoder output shape: (batch size, sequence length, units) {}'.format(sample_output.shape))
print ('Encoder h vecotr shape: (batch size, units) {}'.format(sample_h.shape))
print ('Encoder c vector shape: (batch size, units) {}'.format(sample_c.shape))

Encoder output shape: (batch size, sequence length, units) (32, 25, 500)
Encoder h vecotr shape: (batch size, units) (32, 500)
Encoder c vector shape: (batch size, units) (32, 500)


In [29]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz, max_length_input, attention_type='luong'):
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz
        self.dec_units = dec_units
        self.attention_type = attention_type

        # Embedding Layer
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)

        #Final Dense layer on which softmax will be applied
        self.fc = tf.keras.layers.Dense(vocab_size)

        # Define the fundamental cell for decoder recurrent structure
        self.decoder_rnn_cell = tf.keras.layers.LSTMCell(self.dec_units)



        # Sampler
        self.sampler = tfa.seq2seq.sampler.TrainingSampler()

        # Create attention mechanism with memory = None
        self.attention_mechanism = self.build_attention_mechanism(self.dec_units, 
                                                                  None, self.batch_sz*[max_length_input], self.attention_type)

        # Wrap attention mechanism with the fundamental rnn cell of decoder
        self.rnn_cell = self.build_rnn_cell(batch_sz)

        # Define the decoder with respect to fundamental rnn cell
        self.decoder = tfa.seq2seq.BasicDecoder(self.rnn_cell, sampler=self.sampler, output_layer=self.fc)


    def build_rnn_cell(self, batch_sz):
        rnn_cell = tfa.seq2seq.AttentionWrapper(self.decoder_rnn_cell, 
                                      self.attention_mechanism, attention_layer_size=self.dec_units)
        return rnn_cell

    def build_attention_mechanism(self, dec_units, memory, memory_sequence_length, attention_type='luong'):
        # ------------- #
        # typ: Which sort of attention (Bahdanau, Luong)
        # dec_units: final dimension of attention outputs 
        # memory: encoder hidden states of shape (batch_size, max_length_input, enc_units)
        # memory_sequence_length: 1d array of shape (batch_size) with every element set to max_length_input (for masking purpose)

        if(attention_type=='bahdanau'):
            return tfa.seq2seq.BahdanauAttention(units=dec_units, memory=memory, memory_sequence_length=memory_sequence_length)
        else:
            return tfa.seq2seq.LuongAttention(units=dec_units, memory=memory, memory_sequence_length=memory_sequence_length)

    def build_initial_state(self, batch_sz, encoder_state, Dtype):
        decoder_initial_state = self.rnn_cell.get_initial_state(batch_size=batch_sz, dtype=Dtype)
        decoder_initial_state = decoder_initial_state.clone(cell_state=encoder_state)
        return decoder_initial_state


    def call(self, inputs, initial_state):
        x = self.embedding(inputs)
        outputs, _, _ = self.decoder(x, initial_state=initial_state, sequence_length=self.batch_sz*[max_length_output-1])
        return outputs

In [30]:
decoder = Decoder(vocab_output_size, embedding_dim, hidden_dim, BATCH_SIZE, max_length_input, 'luong')


# Sample
sample_x = tf.random.uniform((BATCH_SIZE, max_length_output))
decoder.attention_mechanism.setup_memory(sample_output)
initial_state = decoder.build_initial_state(BATCH_SIZE, [sample_h, sample_c], tf.float32)


sample_decoder_outputs = decoder(sample_x, initial_state)

print("Decoder Outputs Shape: ", sample_decoder_outputs.rnn_output.shape)

Decoder Outputs Shape:  (32, 25, 43)


In [31]:
encoder.summary()

Model: "encoder_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_2 (Embedding)      multiple                  1050      
_________________________________________________________________
lstm_1 (LSTM)                multiple                  1062000   
Total params: 1,063,050
Trainable params: 1,063,050
Non-trainable params: 0
_________________________________________________________________


In [32]:
decoder.summary()

Model: "decoder_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_3 (Embedding)      multiple                  1290      
_________________________________________________________________
dense_1 (Dense)              multiple                  21543     
_________________________________________________________________
lstm_cell_3 (LSTMCell)       multiple                  2062000   
_________________________________________________________________
LuongAttention (LuongAttenti multiple                  250000    
_________________________________________________________________
attention_wrapper_1 (Attenti multiple                  2812000   
_________________________________________________________________
basic_decoder_1 (BasicDecode multiple                  2833543   
Total params: 2,834,833
Trainable params: 2,834,833
Non-trainable params: 0
_______________________________________________

In [33]:
optimizer = tf.keras.optimizers.Adam()


def loss_function(real, pred):
    # real shape = (BATCH_SIZE, max_length_output)
    # pred shape = (BATCH_SIZE, max_length_output, tar_vocab_size )
    cross_entropy = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
    loss = cross_entropy(y_true=real, y_pred=pred)
    mask = tf.logical_not(tf.math.equal(real,0))   #output 0 for y=0 else output 1
    mask = tf.cast(mask, dtype=loss.dtype)  
    loss = mask* loss
    loss = tf.reduce_mean(loss)
    return loss

In [34]:
@tf.function
def train_step(inp, targ, enc_hidden):
    loss = 0

    with tf.GradientTape() as tape:
        enc_output, enc_h, enc_c = encoder(inp, enc_hidden)


        dec_input = targ[ : , :-1 ] # Ignore <end> token
        real = targ[ : , 1: ]         # ignore <start> token

        # Set the AttentionMechanism object with encoder_outputs
        decoder.attention_mechanism.setup_memory(enc_output)

        # Create AttentionWrapperState as initial_state for decoder
        decoder_initial_state = decoder.build_initial_state(BATCH_SIZE, [enc_h, enc_c], tf.float32)
        pred = decoder(dec_input, decoder_initial_state)
        logits = pred.rnn_output
        loss = loss_function(real, logits)

    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))

    return loss

In [35]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(BATCH_SIZE)
example_input_batch, example_target_batch = next(iter(train_dataset))
example_input_batch.shape, example_target_batch.shape

(TensorShape([32, 25]), TensorShape([32, 26]))

In [62]:
# TO do: keep loss & accuracy for train and validation set


EPOCHS = 2
num_examples = X_train.shape[0]
steps_per_epoch = num_examples//BATCH_SIZE 

for epoch in range(EPOCHS):
    start = time.time()

    enc_hidden = encoder.initialize_hidden_state()
    total_loss = 0
    # print(enc_hidden[0].shape, enc_hidden[1].shape)

    for (batch, (inp, targ)) in enumerate(train_dataset.take(steps_per_epoch)):
        batch_loss = train_step(inp, targ, enc_hidden)
        total_loss += batch_loss

        if batch % 100 == 0:
            print('Epoch {} Batch {} Loss {:.4f}'.format(epoch + 1,
                                                       batch,
                                                       batch_loss.numpy()))
    # saving (checkpoint) the model every 2 epochs
#     if (epoch + 1) % 2 == 0:
#         checkpoint.save(file_prefix = checkpoint_prefix)

    print('Epoch {} Loss {:.4f}'.format(epoch + 1,
                                      total_loss / steps_per_epoch))
    print('Time taken for 1 epoch {} sec\n'.format(time.time() - start))

Epoch 1 Batch 0 Loss 1.2110
Epoch 1 Batch 100 Loss 1.0080
Epoch 1 Batch 200 Loss 0.8607
Epoch 1 Batch 300 Loss 0.6769
Epoch 1 Batch 400 Loss 0.5833
Epoch 1 Batch 500 Loss 0.3770
Epoch 1 Batch 600 Loss 0.2660
Epoch 1 Batch 700 Loss 0.2515
Epoch 1 Batch 800 Loss 0.1747
Epoch 1 Batch 900 Loss 0.1872
Epoch 1 Batch 1000 Loss 0.1847
Epoch 1 Batch 1100 Loss 0.2024
Epoch 1 Batch 1200 Loss 0.1426
Epoch 1 Batch 1300 Loss 0.1540
Epoch 1 Batch 1400 Loss 0.1264
Epoch 1 Batch 1500 Loss 0.1643
Epoch 1 Loss 0.3666
Time taken for 1 epoch 1498.0322196483612 sec

Epoch 2 Batch 0 Loss 0.1434
Epoch 2 Batch 100 Loss 0.1380
Epoch 2 Batch 200 Loss 0.1482
Epoch 2 Batch 300 Loss 0.1214
Epoch 2 Batch 400 Loss 0.1409
Epoch 2 Batch 500 Loss 0.1164
Epoch 2 Batch 600 Loss 0.1174
Epoch 2 Batch 700 Loss 0.1213
Epoch 2 Batch 800 Loss 0.0829
Epoch 2 Batch 900 Loss 0.0996
Epoch 2 Batch 1000 Loss 0.1005
Epoch 2 Batch 1100 Loss 0.1415
Epoch 2 Batch 1200 Loss 0.0950
Epoch 2 Batch 1300 Loss 0.1061
Epoch 2 Batch 1400 Loss 0.0

In [67]:
# For comparison: the basic decoder takes a greedy approach
def greedy_evaluate_sentence(input_sample):
    inputs = tf.convert_to_tensor(input_sample)
    inference_batch_size = inputs.shape[0]
    result = ''

    enc_start_state = [tf.zeros((inference_batch_size, hidden_dim)), tf.zeros((inference_batch_size,hidden_dim))]
    enc_out, enc_h, enc_c = encoder(inputs, enc_start_state)

    dec_h = enc_h
    dec_c = enc_c

    start_tokens = tf.fill([inference_batch_size], conversion_dicts['output_char2idx']['<start>'])
    end_token = conversion_dicts['output_char2idx']['<end>']

    greedy_sampler = tfa.seq2seq.GreedyEmbeddingSampler()

    # Instantiate BasicDecoder object
    decoder_instance = tfa.seq2seq.BasicDecoder(cell=decoder.rnn_cell, sampler=greedy_sampler, output_layer=decoder.fc)
    # Setup Memory in decoder stack
    decoder.attention_mechanism.setup_memory(enc_out)

    # set decoder_initial_state
    decoder_initial_state = decoder.build_initial_state(inference_batch_size, [enc_h, enc_c], tf.float32)


    ### Since the BasicDecoder wraps around Decoder's rnn cell only, you have to ensure that the inputs to BasicDecoder 
    ### decoding step is output of embedding layer. tfa.seq2seq.GreedyEmbeddingSampler() takes care of this. 
    ### You only need to get the weights of embedding layer, which can be done by decoder.embedding.variables[0] and pass this callabble to BasicDecoder's call() function

    decoder_embedding_matrix = decoder.embedding.variables[0]

    outputs, _, _ = decoder_instance(decoder_embedding_matrix, start_tokens = start_tokens, end_token= end_token, initial_state=decoder_initial_state)
    return outputs.sample_id.numpy()


In [68]:
def beam_evaluate_sentence(input_sample, beam_width=3):
    inputs = tf.convert_to_tensor(input_sample)
    inference_batch_size = inputs.shape[0]
    result = ''

    enc_start_state = [tf.zeros((inference_batch_size, hidden_dim)), tf.zeros((inference_batch_size,hidden_dim))]
    enc_out, enc_h, enc_c = encoder(inputs, enc_start_state)

    dec_h = enc_h
    dec_c = enc_c

    start_tokens = tf.fill([inference_batch_size], conversion_dicts['output_char2idx']['<start>'])
    end_token = conversion_dicts['output_char2idx']['<end>']

    # From official documentation
    # NOTE If you are using the BeamSearchDecoder with a cell wrapped in AttentionWrapper, then you must ensure that:
    # The encoder output has been tiled to beam_width via tfa.seq2seq.tile_batch (NOT tf.tile).
    # The batch_size argument passed to the get_initial_state method of this wrapper is equal to true_batch_size * beam_width.
    # The initial state created with get_initial_state above contains a cell_state value containing properly tiled final state from the encoder.

    enc_out = tfa.seq2seq.tile_batch(enc_out, multiplier=beam_width)
    decoder.attention_mechanism.setup_memory(enc_out)
    print("beam_with * [batch_size, max_length_input, rnn_units] :  3 * [1, 16, 1024]] :", enc_out.shape)

    # set decoder_inital_state which is an AttentionWrapperState considering beam_width
    hidden_state = tfa.seq2seq.tile_batch([enc_h, enc_c], multiplier=beam_width)
    decoder_initial_state = decoder.rnn_cell.get_initial_state(batch_size=beam_width*inference_batch_size, dtype=tf.float32)
    decoder_initial_state = decoder_initial_state.clone(cell_state=hidden_state)

    # Instantiate BeamSearchDecoder
    decoder_instance = tfa.seq2seq.BeamSearchDecoder(decoder.rnn_cell,beam_width=beam_width, output_layer=decoder.fc)
    decoder_embedding_matrix = decoder.embedding.variables[0]

    # The BeamSearchDecoder object's call() function takes care of everything.
    outputs, final_state, sequence_lengths = decoder_instance(decoder_embedding_matrix, start_tokens=start_tokens, end_token=end_token, initial_state=decoder_initial_state)
    # outputs is tfa.seq2seq.FinalBeamSearchDecoderOutput object. 
    # The final beam predictions are stored in outputs.predicted_id
    # outputs.beam_search_decoder_output is a tfa.seq2seq.BeamSearchDecoderOutput object which keep tracks of beam_scores and parent_ids while performing a beam decoding step
    # final_state = tfa.seq2seq.BeamSearchDecoderState object.
    # Sequence Length = [inference_batch_size, beam_width] details the maximum length of the beams that are generated


    # outputs.predicted_id.shape = (inference_batch_size, time_step_outputs, beam_width)
    # outputs.beam_search_decoder_output.scores.shape = (inference_batch_size, time_step_outputs, beam_width)
    # Convert the shape of outputs and beam_scores to (inference_batch_size, beam_width, time_step_outputs)
    final_outputs = tf.transpose(outputs.predicted_ids, perm=(0,2,1))
    beam_scores = tf.transpose(outputs.beam_search_decoder_output.scores, perm=(0,2,1))

    return final_outputs.numpy(), beam_scores.numpy()

In [71]:
def beam_convert(sequence, compare_greedy=True):
    inputs = preprocess_sequences([sequence], conversion_dicts['input_char2idx'], max_length_input)
    result, beam_scores = beam_evaluate_sentence(inputs)
    print(result.shape, beam_scores.shape)
    for beam, score in zip(result, beam_scores):
        print(beam.shape, score.shape)
        output = [[conversion_dicts['output_idx2char'][i] for i in b] for b in beam]
        output = [a[:a.index('<end>')] for a in output]
        output = [''.join(a) for a in output]
        beam_score = [a.sum() for a in score]
        print('Input: %s' % (sequence))
        for i in range(len(output)):
            print('{} Predicted translation: {}  {}'.format(i+1, output[i], beam_score[i]))
    if compare_greedy:
        greedy_result = greedy_evaluate_sentence(inputs)
        greedy_result = [[conversion_dicts['output_idx2char'][i] for i in b] for b in greedy_result][0]
        greedy_result = greedy_result[:greedy_result.index('<end>')]
        greedy_result = ''.join(greedy_result)
        print('Greedy result: {}'.format(greedy_result))

In [ ]:
for s_in, s_out in zip(input_seqs_train[:10], output_seqs_train[:10]):
    print('input: {}, output: {}'.format(s_in.strip(), s_out.strip()))
    beam_convert(s_in)
    print('\n')

input: B.EN, output: BN/c
beam_with * [batch_size, max_length_input, rnn_units] :  3 * [1, 16, 1024]] : (3, 25, 500)
(1, 3, 5) (1, 3, 5)
(3, 5) (3, 5)
Input: B.EN
1 Predicted translation: BN/c  -2.5036988258361816
2 Predicted translation: BN  -11.509160041809082
3 Predicted translation: BN/  -16.019039154052734
Greedy result: BN/c


input: MAX:FOP, output: MXFP/
beam_with * [batch_size, max_length_input, rnn_units] :  3 * [1, 16, 1024]] : (3, 25, 500)
(1, 3, 12) (1, 3, 12)
(3, 12) (3, 12)
Input: MAX:FOP
1 Predicted translation: MXFP/  -7.657065391540527
2 Predicted translation: M!(H]XFP[/c  -25.417743682861328
3 Predicted translation: MXFP/a  -35.030174255371094
Greedy result: MXFP/


input: B.;JTOW, output: BJT/+W
beam_with * [batch_size, max_length_input, rnn_units] :  3 * [1, 16, 1024]] : (3, 25, 500)
(1, 3, 9) (1, 3, 9)
(3, 9) (3, 9)
Input: B.;JTOW
1 Predicted translation: BJT/+W  -3.091381549835205
2 Predicted translation: BJT+W  -24.278533935546875
3 Predicted translation: BJT==/